# Munir (منير) — Run All

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lobaali/munir/blob/main/notebook/munir_run_all.ipynb)

**SDAIA Academy Capstone — SDA-AIE-213: LLM Application Engineering**

This notebook is the single entry point for running and evaluating the Munir application.

### Default backend

Munir uses the deterministic mock backend by default, so no API key is required.

### Run

Use **Kernel → Restart Kernel and Run All** to execute the complete evaluation.

The notebook runs:

1. Automated tests
2. Golden-set evaluation
3. Judge calibration
4. Regression gate
5. Cost and latency replay
6. Commercial vs open-weight comparison
7. Self-host break-even analysis

The generated evaluation artifacts are written to the repository's `eval/out/` directory.

In [1]:
# ============================================================
# 1. Environment setup (Colab-aware)
# ============================================================
# If this notebook is opened fresh via the Colab badge, the repository
# has not been cloned yet -- this cell detects that and clones it.
# If it is opened from an existing local checkout, it just locates the
# repo root, exactly as before.

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Lobaali/munir.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd()

# Walk up looking for an existing repo root.
search = ROOT
while search != search.parent and not (search / "src" / "munir").exists():
    search = search.parent

if (search / "src" / "munir").exists():
    ROOT = search

elif IN_COLAB:
    # Fresh Colab runtime: the repo is not on disk yet -- clone it.
    clone_dir = Path("/content/munir")

    if not (clone_dir / "src" / "munir").exists():
        print(f"Cloning {REPO_URL} into {clone_dir} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
            check=True,
        )

    ROOT = clone_dir

else:
    raise RuntimeError(
        "Could not find the Munir repository root, and this does not look "
        "like a Colab runtime. Open this notebook from inside the Munir "
        "repository, or run it via the Colab badge in README.md."
    )

os.chdir(ROOT)

# Make both the repository root and src/ available to Python.
current_pythonpath = os.environ.get("PYTHONPATH", "")
pythonpath_parts = [str(ROOT), str(ROOT / "src")]

if current_pythonpath:
    pythonpath_parts.append(current_pythonpath)

os.environ["PYTHONPATH"] = os.pathsep.join(pythonpath_parts)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repository: {ROOT}")
print(f"Python: {sys.executable}")
print(f"PYTHONPATH: {os.environ['PYTHONPATH']}")


Cloning https://github.com/Lobaali/munir.git into /content/munir ...
Environment: Colab
Repository: /content/munir
Python: /usr/bin/python3
PYTHONPATH: /content/munir:/content/munir/src:/env/python


In [2]:
# ============================================================
# 2. Verify project structure
# ============================================================

required_paths = [
    Path("src/munir"),
    Path("configs/munir.yaml"),
    Path("data"),
    Path("eval"),
    Path("scripts"),
    Path("tests"),
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
]

missing = [str(path) for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The following required project paths are missing:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

print("✓ Munir project structure verified.")

✓ Munir project structure verified.


In [3]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
requirements = ROOT / "requirements.txt"

if not requirements.exists():
    raise FileNotFoundError(f"Could not find {requirements}")

print(f"Installing project dependencies from: {requirements}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)

print("\nRunning automated test suite...\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The automated test suite failed. "
        "Review the pytest output above before continuing."
    )

print("\n✓ Automated test suite passed.")

Installing project dependencies from: /content/munir/requirements.txt

Running automated test suite...

......................................                                   [100%]
38 passed in 1.01s


✓ Automated test suite passed.


In [4]:
# ============================================================
# 4. Run the evaluation pipeline
# ============================================================

commands = [
    (
        "Golden-set evaluation",
        [sys.executable, "eval/harness.py"],
    ),
    (
        "Judge calibration",
        [sys.executable, "eval/calibrate_judge.py"],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("\n--- STDERR ---")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )


# ------------------------------------------------------------
# Regression gate
# The harness above generates eval/out/eval_run.json.
# We pass that freshly generated report to the gate.
# ------------------------------------------------------------

report_path = Path("eval/out/eval_run.json")

if not report_path.exists():
    raise FileNotFoundError(
        f"Expected evaluation report was not generated: {report_path}"
    )

print("\n" + "=" * 70)
print("Regression gate")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        "eval/gate.py",
        str(report_path),
    ],
    check=False,
    capture_output=True,
    text=True,
)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Regression gate failed with exit code {result.returncode}."
    )

print("\n✓ Evaluation pipeline completed successfully.")


Golden-set evaluation

----------------------------------------------------------------------------
eval | route=default | 120 cases | pass 120/120 (100%) | 0.38s
----------------------------------------------------------------------------
  language    ar 100% | en 100%
  intent      escalate 100% | faq 100% | service 100%
  difficulty  adversarial 100% | edge 100% | routine 100%
  risk        normal 100% | safety 100%
  p50 latency 3.2 ms

  written: /content/munir/eval/out/eval_run.json


--- STDERR ---
2026-09-15T12:03:55.731160Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=461ba58645ca
2026-09-15T12:03:55.731565Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00297 input_tokens=178 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=461ba58645ca
2026-09-15T12:03:55.732772Z [info     ] structured_extracted           

In [5]:
# ============================================================
# 5. Run cost, cache, model-comparison, and break-even evidence
# ============================================================

commands = [
    (
        "Cost and latency replay",
        [
            sys.executable,
            "scripts/replay.py",
            "--limit",
            "120",
            "--write",
        ],
    ),
    (
        "Commercial vs open-weight comparison",
        [
            sys.executable,
            "scripts/compare_models.py",
            "--limit",
            "120",
        ],
    ),
    (
        "Self-host break-even analysis",
        [
            sys.executable,
            "scripts/breakeven.py",
            "--gpu-usd-per-hour",
            "3.33",
            "--tokens-per-sec",
            "950",
            "--utilization",
            "0.50",
            "--commercial-price-per-mtok",
            "15",
            "--avg-tokens-per-request",
            "100",
        ],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )

print("\n✓ Cost, model comparison, and break-even analysis completed.")


Cost and latency replay

Commercial vs open-weight comparison

Self-host break-even analysis

✓ Cost, model comparison, and break-even analysis completed.


In [6]:
# ============================================================
# 6. Verify generated evaluation artifacts
# ============================================================

print("=" * 70)
print("GENERATED EVIDENCE")
print("=" * 70)

files_to_check = [
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
    Path("eval/out/cost_optimization_comparison.json"),
    Path("eval/out/commercial_vs_open_weight.json"),
]

for path in files_to_check:
    if path.exists():
        print(f"✓ {path}")
    else:
        print(f"⚠ Missing: {path}")

print("\nRun All completed.")

GENERATED EVIDENCE
✓ BENCHMARKS.md
✓ EVALUATION_REPORT.md
✓ DECISIONS.md
✓ eval/out/cost_optimization_comparison.json
✓ eval/out/commercial_vs_open_weight.json

Run All completed.
